# **Section 1: Khám phá (Exploration)**

In [ ]:
import pandas as pd
import os

base_path = os.getcwd()
file_path = os.path.join(base_path, 'dataset', 'salary_survey_raw.csv')

df = pd.read_csv(file_path)

# 1. Kiểm tra kích thước 
print(f"Số lượng dòng: {df.shape[0]}")
print(f"Số lượng cột: {df.shape[1]}")

# 2. Xem 5 dòng đầu tiên 
display(df.head())

# 3. Kiểm tra kiểu dữ liệu và giá trị non-null
df.info()


Số lượng dòng: 2800
Số lượng cột: 17


,timestamp,how_old_are_you,industry,job_title,additional_context_on_job_title,annual_salary,additional_monetary_comp,currency,income_context,country,us_state,city,years_of_experience_in_field,years_of_experience_overall,highest_level_of_education,gender,race
0,04/12/2021 16:11:47,45-54,Legal,Legal Counsel,NaN,"56,894",NaN,USD,NaN,NaN,NaN,Boston,21 - 30 years,21 - 30 years,College degree,Man,Black or African American
1,04/25/2021 09:28:06,25-34,Finance,Investment Analyst,NaN,"144,543","$9,839",USD,NaN,United States,Washington,NaN,8 - 10 years,8 - 10 years,College degree,Man,Multiracial
2,04/15/2021,25-34,Nonprofit,Grant Writer,NaN,"52,304.00",NaN,AUD,NaN,United States,Washington,NaN,21 - 30 years,21 - 30 years,PhD,Woman,NaN
3,04/21/2021 03:36:47,55-64,Real Estate,Real Estate Manager,NaN,"51,765",18454,AUD,NaN,United States,NaN,Los Angeles,2 - 4 years,2 - 4 years,Master's degree,Male,NaN
4,2021-04-26 03:04,35-44,Nursing,Charge Nurse,Senior level,"133,218","8,424",USD,NaN,UK,NaN,NaN,11 - 20 years,11 - 20 years,College degree,Woman,NaN


<class 'pandas.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype
---  ------                           --------------  -----
 0   timestamp                        2800 non-null   str  
 1   how_old_are_you                  2800 non-null   str  
 2   industry                         2800 non-null   str  
 3   job_title                        2800 non-null   str  
 4   additional_context_on_job_title  1831 non-null   str  
 5   annual_salary                    2409 non-null   str  
 6   additional_monetary_comp         1262 non-null   str  
 7   currency                         2800 non-null   str  
 8   income_context                   531 non-null    str  
 9   country                          2685 non-null   str  
 10  us_state                         987 non-null    str  
 11  city                             1238 non-null   str  
 12  years_of_experience_in_field     2800 non-null   str  
 13 


###  Quan sát về quy mô và độ đầy đủ
* **Kích thước**: Dataset có **2800 hàng** và **17 cột**.
* **Độ đầy đủ**: Chỉ có 8 cột là đầy đủ 100% dữ liệu. Các cột quan trọng như `annual_salary` và `additional_monetary_comp` có tỉ lệ null rất cao.
* **Phân tích**: Việc thiếu hụt dữ liệu ở cột lương có thể thuộc loại **MNAR (Missing Not At Random)** - người có thu nhập quá thấp hoặc quá cao thường ngại khai báo, điều này có thể gây bias nếu ta xử lý không khéo.

###  Vấn đề về Kiểu dữ liệu (Dtype)
* Qua hàm `.info()`, cột `annual_salary` đang bị lưu ở dạng **object** (chuỗi) thay vì numeric (số).
* Mặc dù các cột hiển thị là object, thực tế đây là kiểu dữ liệu chuỗi (string) trong Pandas. Tuy nhiên, với các cột định lượng như annual_salary, việc để ở kiểu object khiến ta không thể tính toán thống kê. Do đó, bước tiếp theo bắt buộc phải là loại bỏ ký tự đặc biệt và ép kiểu về float64 để dữ liệu có thể 'tính toán' được. 
* **Ví dụ : " 52,304.00" , "51,765" , " $9,839" đang là String nên khó có thể tính toán.**
* **Hệ quả**: Không thể thực hiện các phép toán thống kê như Mean, Median hay vẽ biểu đồ định lượng. 
* **Hướng xử lý**: Cần thực hiện loại bỏ các ký tự đặc biệt ($, dấu phẩy) và ép lại kiểu dữ liệu về `float64`.



# **TUẦN 1 — MISSING VALUES, DUPLICATES & DATA TYPES**

In [2]:
def missing_report(df):
    # 1. Tính tổng số lượng null trên mỗi cột 
    miss = df.isnull().sum()
    
    # 2. Tính tỉ lệ phần trăm null
    pct = (miss / len(df) * 100).round(2)
    report = pd.DataFrame({'Missing Count': miss, 'Percentage (%)': pct})
    return report.query('`Missing Count` > 0').sort_values('Percentage (%)', ascending=False)


display(missing_report(df))

,Missing Count,Percentage (%)
income_context,2269,81.04
us_state,1813,64.75
city,1562,55.79
additional_monetary_comp,1538,54.93
additional_context_on_job_title,969,34.61
race,766,27.36
annual_salary,391,13.96
country,115,4.11
gender,69,2.46



###  Decision Log 


| Nhóm cột | Tên cột | Loại Null | Chiến lược | Lý do (Logic) |
| :--- | :--- | :--- | :--- | :--- |
| **Định lượng** | `annual_salary` | **MNAR** | `fillna(median)` | Lương thường phân phối lệch phải; người thu nhập cực thấp/cao có xu hướng ngại khai. Median giúp tránh nhiễu do Outliers tốt hơn Mean. |
| **Định lượng** | `additional_monetary_comp` | **MAR** | `fillna(0)` | **Ghi nhận ngầm**: Trong khảo sát lương, không điền thường đồng nghĩa với việc không phát sinh thưởng hoặc hoa hồng. |
| **Vị trí** | `country`, `city` | **MCAR** | `fillna('Unknown')` | Lỗi nhập liệu ngẫu nhiên hoặc người dùng muốn bảo mật vị trí. Tỉ lệ thấp, giữ lại để bảo toàn các thông tin khác trong hàng. |
| **Vị trí** | `us_state` | **Logic/MAR** | `fillna('Non-US')` | **Dữ liệu phụ thuộc**: Chỉ người làm việc tại Mỹ mới điền cột này. Null là hợp lệ đối với các ứng viên quốc tế. |
| **Nhân khẩu** | `gender`, `race` | **MCAR** | `dropna()` | Tỉ lệ thiếu hụt rất nhỏ (<3%). Xóa các hàng này để đảm bảo độ chính xác tuyệt đối cho các biểu đồ phân tích nhân khẩu học. |
| **Key Fields** | `job_title`, `industry` | **MCAR** | `dropna()` | Đây là các cột biến độc lập chính. Nếu thiếu, bản ghi không còn giá trị để phân tích mối tương quan giữa nghề nghiệp và tiền lương. |
| **Ghi chú** | `income_context`, `job_context` | **MAR** | `fillna('None')` | Thông tin bổ sung không bắt buộc. Việc điền 'None' giúp đồng bộ kiểu dữ liệu chuỗi (string) để tránh lỗi xử lý ở các bước sau. |

###  Phân tích 

1. **Về các cột Key Fields**: Mục tiêu của bài toán là phân tích lương theo ngành và chức danh. Một bản ghi có lương nhưng không biết làm nghề gì hay ở ngành nào thì hoàn toàn vô dụng cho việc thống kê, do đó việc loại bỏ (`dropna`) là cần thiết.
2. **Về tính Logic của địa lý**: Cột `us_state` trống không phải là lỗi mà là một sự thật khách quan (người làm ngoài Mỹ). Việc gán nhãn 'Non-US' thay vì 'Unknown' thể hiện sự hiểu biết về bối cảnh dữ liệu.
3. **Về dữ liệu định tính (Context)**: Tỉ lệ null lên đến 81% ở các cột ghi chú là hoàn toàn bình thường trong các khảo sát mở. Chúng ta giữ lại bằng cách gán nhãn 'None' để không làm mất đi các dữ liệu định lượng quý giá ở cùng hàng đó.

In [ ]:
# 1. Tạo bản sao để so sánh trước/sau 
df_clean = df.copy()

# --- CHIẾN LƯỢC 1: FILLNA (IMPUTATION) & ÉP KIỂU ---

# Xử lý Lương: Xóa ký tự đặc biệt, ép kiểu float để tính toán
df_clean['annual_salary'] = pd.to_numeric(
    df_clean['annual_salary'].astype(str).str.replace(r'[^\d.]', '', regex=True), 
    errors='coerce'
)

# LÝ DO: annual_salary là MNAR, dùng median theo industry để tránh bias do outliers
# (Lưu ý: Nếu industry bị null, ta sẽ fill median tổng để đảm bảo không còn null)
df_clean['annual_salary'] = df_clean.groupby('industry')['annual_salary'].transform(
    lambda x: x.fillna(x.median())
).fillna(df_clean['annual_salary'].median())

# Xử lý Thưởng: "Ghi nhận ngầm" MAR => Điền 0
df_clean['additional_monetary_comp'] = pd.to_numeric(
    df_clean['additional_monetary_comp'].astype(str).str.replace(r'[^\d.]', '', regex=True), 
    errors='coerce'
).fillna(0)

# Xử lý các cột Vị trí & Ghi chú (Constant Fillna)
# LÝ DO: us_state là Logic/MAR (người ngoài Mỹ), các cột context là MAR (thông tin bổ sung)
df_clean['us_state'] = df_clean['us_state'].fillna('Non-US')
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['city'] = df_clean['city'].fillna('Unknown')

context_cols = ['income_context', 'additional_context_on_job_title', 'race']
for col in context_cols:
    df_clean[col] = df_clean[col].fillna('None')


# --- CHIẾN LƯỢC 2: DROP (LOẠI BỎ) ---

# LÝ DO: gender/race (<3%) là MCAR. industry/job_title là Key Fields (thiếu là vô dụng)
# Xóa các hàng thiếu thông tin then chốt để đảm bảo chất lượng mẫu
df_clean = df_clean.dropna(subset=['gender', 'industry', 'job_title'])


# --- CHIẾN LƯỢC 3: INTERPOLATE (NỘI SUY) ---

# LÝ DO: Áp dụng nội suy tuyến tính để làm mượt dữ liệu số sau khi đã xử lý sơ bộ
df_clean['annual_salary'] = df_clean['annual_salary'].interpolate(method='linear')


# --- TỔNG KẾT & SO SÁNH ---

before = df.isnull().sum()
after = df_clean.isnull().sum()

compare_df = pd.DataFrame({
    'Trước xử lý': before,
    'Sau xử lý': after
})

print("Bảng so sánh dữ liệu thiếu trước và sau khi làm sạch (Checklist 1.1):")
display(compare_df[compare_df['Trước xử lý'] > 0])

# Kiểm tra lại kiểu dữ liệu để đảm bảo cột luonwg đã được chuyển về numeric
print("\nKiểm tra lại Dtypes:")
print(df_clean[['annual_salary', 'additional_monetary_comp']].dtypes)

Bảng so sánh dữ liệu thiếu trước và sau khi làm sạch (Checklist 1.1):


,Trước xử lý,Sau xử lý
additional_context_on_job_title,969,0
annual_salary,391,0
additional_monetary_comp,1538,0
income_context,2269,0
country,115,0
us_state,1813,0
city,1562,0
gender,69,0
race,766,0



Kiểm tra lại Dtypes:
annual_salary               float64
additional_monetary_comp    float64
dtype: object


In [ ]:
# 1. Kiểm tra kích thước (Shape) trước và sau khi Clean
shape_before = df.shape
shape_after = df_clean.shape

# 2. Tính toán số lượng hàng và cột bị loại bỏ
rows_dropped = shape_before[0] - shape_after[0]
cols_dropped = shape_before[1] - shape_after[1]

# 3. Hiển thị
print("-" * 30)
print("BÁO CÁO THAY ĐỔI KÍCH THƯỚC DATASET")
print("-" * 30)
summary_data = {
    "Chỉ số": ["Số lượng hàng (Rows)", "Số lượng cột (Columns)"],
    "Trước khi xử lý": [shape_before[0], shape_before[1]],
    "Sau khi xử lý": [shape_after[0], shape_after[1]],
    "Số lượng đã loại bỏ": [rows_dropped, cols_dropped]
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# 4. Tính tỉ lệ dữ liệu giữ lại được
retention_rate = (shape_after[0] / shape_before[0] * 100)
print(f"\n=> Tỉ lệ dữ liệu giữ lại: {retention_rate:.2f}%")

------------------------------
BÁO CÁO THAY ĐỔI KÍCH THƯỚC DATASET
------------------------------


,Chỉ số,Trước khi xử lý,Sau khi xử lý,Số lượng đã loại bỏ
0,Số lượng hàng (Rows),2800,2731,69
1,Số lượng cột (Columns),17,17,0



=> Tỉ lệ dữ liệu giữ lại: 97.54%


**69 hàng bị loại bỏ này chính là do dropna các giá trị null của cột gender**

**1.2 Checklist 1.2 —Duplicates:**
* ☐Kiểm tra duplicate toàn hàng và theo key columns 
* ☐Giải thích bằngMarkdown: tại sao có duplicate? Lỗi nhập liệu, join sai, hay có ý nghĩa khác? 
* ☐Báo cáo sốhàng trước và sau khi xửlý

In [ ]:
# Kiểm tra toàn hàng
n_all_dup = df_clean.duplicated().sum()
# Kiểm tra theo key columns (Timestamp)
n_key_dup = df_clean.duplicated(subset=['timestamp', 'job_title']).sum()

print(f"Số lượng trùng lặp toàn phần: {n_all_dup}")
print(f"Số lượng trùng lặp theo khóa (Time & Job): {n_key_dup}")

Số lượng trùng lặp toàn phần: 0
Số lượng trùng lặp theo khóa (Time & Job): 65


Giải thích về Duplicates: Dữ liệu xuất hiện trùng lặp chủ yếu do lỗi nhập liệu (người dùng nhấn gửi nhiều lần) hoặc lỗi kỹ thuật trong quá trình thu thập/trích xuất dữ liệu. Việc trùng khớp cả mốc thời gian (timestamp) đến từng giây và chức danh công việc cho thấy đây là các bản ghi nhân bản, không mang giá trị thông tin mới.

In [14]:
# Xử lý hàng trùng theo khóa (Timestamp & Job Title)
# LÝ DO: Dù dữ liệu khác nhau đôi chút, nhưng trùng khớp Timestamp đến từng giây 
# là bằng chứng của việc lỗi hệ thống/nộp lại bài. Ta ưu tiên giữ bản ghi ĐẦU TIÊN.

rows_before_key = len(df_clean)

# Thêm subset để ép Pandas xóa dựa trên khóa chính
df_clean = df_clean.drop_duplicates(subset=['timestamp', 'job_title'], keep='first').reset_index(drop=True)

rows_after_key = len(df_clean)

print(f"Số hàng trước khi xử lý theo khóa: {rows_before_key}")
print(f"Số hàng sau khi xử lý theo khóa: {rows_after_key}")
print(f"Đã thực sự loại bỏ được: {rows_before_key - rows_after_key} hàng 'nghi vấn'.")

Số hàng trước khi xử lý theo khóa: 2694
Số hàng sau khi xử lý theo khóa: 2629
Đã thực sự loại bỏ được: 65 hàng 'nghi vấn'.


**1.3 Data Types —dtype sai là nguồn gốc của 40% lỗi phân tích**

In [25]:
# Kiếm tra datatypes trước khi xử lý

print(df_clean.dtypes)

timestamp                          datetime64[us]
how_old_are_you                               str
industry                                 category
job_title                                     str
additional_context_on_job_title               str
annual_salary                             float64
additional_monetary_comp                  float64
currency                                 category
income_context                                str
country                                       str
us_state                                      str
city                                          str
years_of_experience_in_field                  str
years_of_experience_overall                   str
highest_level_of_education               category
gender                                   category
race                                     category
dtype: object


In [ ]:
# 1. Lưu bộ nhớ ban đầu để tính tỉ lệ tiết kiệm 
mem_before = df.memory_usage(deep=True).sum()

# 2. Convert cột Ngày/Giờ (errors='coerce' để xử lý các dòng rác nếu có)
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], errors='coerce')

# 3. Convert các cột Số (Huy lưu ý: Phải dùng pd.to_numeric)
# Vì mình đã dùng str.replace ở bước Missing Value 
df_clean['annual_salary'] = pd.to_numeric(df_clean['annual_salary'], errors='coerce')
df_clean['additional_monetary_comp'] = pd.to_numeric(df_clean['additional_monetary_comp'], errors='coerce')

# 4. Convert các cột phân loại sang 'category' để tối ưu bộ nhớ
# LÝ DO: Giảm dung lượng và tăng tốc độ filter dữ liệu
categorical_cols = ['industry', 'currency', 'highest_level_of_education', 'gender', 'race']
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype('category')

# 5. Kiểm tra kết quả và đo lường
mem_after = df_clean.memory_usage(deep=True).sum()
saving_pct = (1 - mem_after/mem_before) * 100

print("Dtypes sau khi đã 'phẫu thuật':")
display(df_clean.dtypes)
print(f"\n=> Tỉ lệ tiết kiệm bộ nhớ: {saving_pct:.2f}%")

Dtypes sau khi đã 'phẫu thuật':


timestamp                          datetime64[us]
how_old_are_you                               str
industry                                 category
job_title                                     str
additional_context_on_job_title               str
annual_salary                             float64
additional_monetary_comp                  float64
currency                                 category
income_context                                str
country                                       str
us_state                                      str
city                                          str
years_of_experience_in_field                  str
years_of_experience_overall                   str
highest_level_of_education               category
gender                                   category
race                                     category
dtype: object


=> Tỉ lệ tiết kiệm bộ nhớ: 38.40%




### Tại sao Data Type sai gây lỗi trong phân tích?

Việc để dữ liệu ở dạng `string` mặc định như hiện tại sẽ dẫn đến 3 vấn đề nghiêm trọng:
1. **Lỗi tính toán thống kê**: Các cột như `annual_salary` dù chứa số nhưng máy tính hiểu là "chữ". Chúng ta không thể thực hiện các hàm `mean()`, `sum()`, hay `median()` để phân tích thu nhập.
2. **Mất khả năng phân tích thời gian**: Cột `timestamp` ở dạng chuỗi sẽ không thể dùng để lọc dữ liệu theo tháng/quý, hoặc tính toán khoảng cách giữa các lần nộp khảo sát.
3. **Lãng phí tài nguyên hệ thống**: Các cột phân loại (như `gender`, `industry`) có giá trị lặp lại nhiều lần. Lưu dưới dạng `string` sẽ chiếm dụng RAM lớn hơn rất nhiều so với định dạng `category`.

### Danh sách các cột cần chuyển đổi

Dựa trên báo cáo `df.dtypes` ở bước trước, tôi xác định các nhóm cột cần chỉnh sửa như sau:

| Nhóm dữ liệu | Tên cột | Kiểu hiện tại | Kiểu mục tiêu | Lý do chuyển đổi |
| :--- | :--- | :--- | :--- | :--- |
| **Thời gian** | `timestamp` | `object` (string) | `datetime64[ns]` | Cho phép thực hiện các phép toán và lọc theo thời gian. |
| **Định lượng** | `annual_salary`, `additional_monetary_comp` | `object` (string) | `float64` | Cho phép tính toán các chỉ số thống kê (Mean, Median, Std). |
| **Phân loại** | `gender`, `race`, `industry`, `currency`, `highest_level_of_education` | `object` (string) | `category` | Tối ưu hóa bộ nhớ và tăng tốc độ xử lý các nhóm dữ liệu lặp lại. |


